# 02 — Precipitation Interpolation

Three-step workflow:

| Step | Parameter | This notebook |
|------|-----------|---------------|
| 1 | `data=` | Open-Meteo archive grid (or offline synthetic) |
| 2 | `boundary=` | Named place or 4-corner bbox |
| 3 | `method=` | IDW · Kriging · RBF · Spline |

**Offline cells** run without any network access.  
**🌐 Network cells** fetch live Open-Meteo data.  
**🗺 Interactive maps** use [geemap](https://geemap.org).

In [ ]:
%pip install -q "geointerpo[full]" geemap localtileserver

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import pandas as pd
from geointerpo import Pipeline

## Configuration

In [ ]:
BOUNDARY   = (-120.0, 49.0, -110.0, 60.0)  # Alberta, Canada  (min_lon, min_lat, max_lon, max_lat)
PLACE      = 'Alberta, Canada'              # 🌐 resolves to actual province boundary via Nominatim
DATE       = '2024-04-15'
RESOLUTION = '25km'  # human-readable resolution string; also accepts float degrees e.g. 0.25

---
## Part A — Offline demo (no network needed)

In [ ]:
result = Pipeline(
    data='sample',                # Step 1 — built-in synthetic precipitation data
    variable='precipitation',
    boundary=BOUNDARY,            # Step 2 — four corners
    method=['idw', 'kriging', 'rbf', 'spline'],  # Step 3
    method_params={
        'idw':    {'power': 2},
        'kriging': {'variogram_model': 'gaussian'},
        'rbf':    {'kernel': 'thin_plate_spline'},
    },
    resolution=RESOLUTION,
    cv_folds=5,
).run()

In [ ]:
print('Cross-validation metrics:')
print(result.metrics_table())
print(f'\nBest method by RMSE: {result.best_method()}')
result.rank_methods()

In [ ]:
result.plot(cmap='Blues')
plt.suptitle('Precipitation — Alberta, Canada (synthetic demo)', y=1.02)
plt.show()

### Station observations

In [ ]:
ax = result.stations.plot(
    column='value', cmap='Blues', legend=True,
    figsize=(9, 6), markersize=40, edgecolor='k', linewidth=0.4
)
ax.set_title('Precipitation stations — Alberta (synthetic, mm/day)')
plt.tight_layout()

### Effect of network sparsity

Re-run with progressively fewer stations to see how CV metrics degrade.

In [ ]:
from geointerpo.data.samples import load_precipitation
from geointerpo.interpolators import KrigingInterpolator

full_gdf = load_precipitation(bbox=BOUNDARY)
rows = []
for n in [50, 30, 20, 15, 10]:
    sparse = full_gdf.sample(min(n, len(full_gdf)), random_state=42).reset_index(drop=True)
    model = KrigingInterpolator(variogram_model='gaussian').fit(sparse)
    cv = model.cross_validate(sparse, k=5)
    rows.append({'n_stations': n, **{k: round(v, 3) for k, v in cv.items() if k != 'n'}})

pd.DataFrame(rows).set_index('n_stations')

### IDW power (β) sensitivity

A [Towards Data Science analysis of spatial interpolation](https://towardsdatascience.com/3-best-methods-for-spatial-interpolation-912cab7aee47/) highlights that IDW's β parameter controls how steeply influence drops with distance. β = 1 gives linear decay (smooth, regional); higher β values create sharper "bull's-eye" patterns centred on each station. The optimal β minimises CV error for a given dataset density.

In [ ]:
from geointerpo.interpolators import IDWInterpolator
import pandas as pd

rows = []
for beta in [1, 2, 3, 4, 5]:
    idw = IDWInterpolator(power=beta).fit(result.stations)
    cv  = idw.cross_validate(result.stations, k=5)
    rows.append({'beta (power)': beta, 'rmse': round(cv['rmse'], 3), 'r': round(cv['r'], 3)})

df_beta = pd.DataFrame(rows).set_index('beta (power)')
print('IDW cross-validation vs power β (precipitation, 5-fold spatial CV):')
df_beta

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, beta in zip(axes, [1, 2, 5]):
    idw = IDWInterpolator(power=beta).fit(result.stations)
    grid = idw.predict(BOUNDARY, resolution=RESOLUTION)
    grid.plot(ax=ax, cmap='Blues')
    result.stations.plot(ax=ax, color='k', markersize=15, zorder=5)
    ax.set_title(f'IDW β = {beta}')

plt.suptitle('IDW power sensitivity — precipitation Alberta (synthetic, mm/day)', y=1.02)
plt.tight_layout()
plt.show()

---
## Part B — 🌐 Live Open-Meteo data

Open-Meteo provides a free archive grid — no API key required.

In [ ]:
result_live = Pipeline(
    data='openmeteo',
    variable='precipitation',
    date=DATE,
    boundary=PLACE,                # clips to actual Alberta polygon
    method=['idw', 'kriging', 'rbf'],
    method_params={
        'kriging': {'variogram_model': 'gaussian'},
    },
    resolution=RESOLUTION,
    cv_folds=5,
).run()

print(f'Loaded {len(result_live.stations)} grid points')

In [ ]:
# Sub-sample to simulate a sparse rain-gauge network
sparse_gdf = result_live.stations.sample(25, random_state=42).reset_index(drop=True)
print(f'Sparse network: {len(sparse_gdf)} stations')

from geointerpo.interpolators import KrigingInterpolator
from geointerpo.boundaries import boundary_bbox
from geointerpo.boundaries import load_boundary

bnd = load_boundary(BOUNDARY)
bbox = boundary_bbox(bnd)
model = KrigingInterpolator(variogram_model='gaussian').fit(sparse_gdf)
grid  = model.predict(bbox, resolution=RESOLUTION)
cv    = model.cross_validate(sparse_gdf, k=5)
print(f'Sparse Kriging CV  RMSE={cv["rmse"]:.2f}  r={cv["r"]:.3f}')

In [ ]:
result_live.plot(cmap='Blues')
plt.suptitle(f'Precipitation — Alberta {DATE}  (Open-Meteo)', y=1.02)
plt.show()

### Interactive map

### City-level zoom — Edmonton & Calgary

Clip precipitation to city boundaries to see local variation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, city in zip(axes, ['Edmonton, Alberta', 'Calgary, Alberta']):
    city_result = Pipeline(
        data='sample',
        variable='precipitation',
        boundary=city,
        method='kriging',
        method_params={'kriging': {'variogram_model': 'gaussian'}},
        resolution=0.05,
        cv_folds=3,
    ).run()
    city_result.grid.plot(ax=ax, cmap='Blues')
    city_result.stations.plot(ax=ax, color='k', markersize=20, zorder=5)
    ax.set_title(city.split(',')[0])

plt.suptitle('Precipitation — city-level zoom (synthetic, mm/day)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Interactive map — requires: pip install geointerpo[interactive]
try:
    fig = result.plot_interactive(backend='auto')
    fig  # renders inline in Jupyter
except ImportError:
    print("Install plotly for interactive maps: pip install 'geointerpo[interactive]'")

---
## Save outputs

In [ ]:
result.save('outputs/precipitation', geotiff=True, plot=True)
print('Saved to outputs/precipitation/')